# Arabic Multimodal QA on OrangePi AI Pro

这个 Notebook 整理为**可提交版一体化流程**，包含：

- 环境验证
- 依赖安装
- 数据下载
- 模型下载
- 使用**相对路径**组织项目
- 生成 Audio / Image 两类 manifest
- 加载 MindSpore + MindNLP 模型
- 进行 Audio / Image 两类问答测试

## 目标
面向 **WanJuanSiLu2O 阿拉伯语多模态数据资源**，实现：

1. **Audio-Text QA**：基于音频文件及其阿语转写文本进行问答  
2. **Image-Text QA**：基于图片 URL 及其阿语说明文本进行问答  

## 说明
- 本 Notebook 默认采用**相对路径**
- 为了避免误下载大量数据，安装/下载 cell 使用开关控制
- 你可以先运行前几节完成环境准备，再运行后面的模型和应用部分


## 0. 一键运行说明

建议按以下顺序执行：

1. 运行 **项目根目录与相对路径** cell  
2. 运行 **环境验证** cell  
3. 按需把 `INSTALL_MISSING`、`DOWNLOAD_MODEL`、`DOWNLOAD_DATA` 改为 `True`  
4. 运行 **依赖安装 / 下载** cells  
5. 运行 **生成 manifest** cells  
6. 运行 **模型加载与测试** cells  
7. 最后运行 **导出 `multimodal_qa_web.py`** cell

### 目录建议
建议把 Notebook 放在如下目录：

```text
Online/arabic_multimodal_qa/
├─ arabic_multimodal_qa_all_in_one.ipynb
├─ model/
│  └─ Qwen2-0.5B-Instruct/
├─ data/
│  └─ WanJuanSiLu2O_ar/
├─ outputs/
│  ├─ audio_demo/
│  └─ image_demo/
```

### OpenXLab 登录说明
如果需要在 Notebook 里下载数据集，可以提前在终端执行：

```bash
openxlab login
```

或者在后面的 cell 中设置环境变量 `OPENXLAB_AK` / `OPENXLAB_SK` 后自动登录。

### 参考说明
OpenXLab 的 CLI 可通过 `pip install openxlab` 安装，并支持 `openxlab dataset download --dataset-repo ... --source-path ... --target-path ...` 这种下载方式。citeturn0search2turn0search12

MindSpore 的 `pynative_synchronize=True` 可以让 PyNative 模式切到同步执行，便于定位设备端报错，但会降低性能。citeturn0search1turn0search13


## 1. 项目根目录与相对路径
这一节统一定义项目目录，后续所有代码都基于相对路径，不再写死绝对路径。

In [1]:

from pathlib import Path
import os
import sys
import json
import subprocess
import shutil

# 假设 notebook 位于项目根目录内执行
PROJECT_ROOT = Path.cwd()

DATA_ROOT = PROJECT_ROOT / "data" / "WanJuanSiLu2O_ar"
MODEL_ROOT = PROJECT_ROOT / "model" / "Qwen2-0.5B-Instruct"
OUTPUT_ROOT = PROJECT_ROOT / "outputs"
AUDIO_OUT = OUTPUT_ROOT / "audio_demo"
IMAGE_OUT = OUTPUT_ROOT / "image_demo"

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
AUDIO_OUT.mkdir(parents=True, exist_ok=True)
IMAGE_OUT.mkdir(parents=True, exist_ok=True)

# 常用路径
RAW_AUDIO_ROOT = DATA_ROOT / "OpenDataLab___WanJuanSiLu2O" / "raw" / "audio" / "ar"
RAW_IMAGE_ROOT = DATA_ROOT / "OpenDataLab___WanJuanSiLu2O" / "raw" / "image" / "ar"

AUDIO_MANIFEST = AUDIO_OUT / "extracted_manifest_multimodal.jsonl"
IMAGE_MANIFEST = IMAGE_OUT / "image_manifest_url.jsonl"

print("PROJECT_ROOT =", PROJECT_ROOT)
print("DATA_ROOT    =", DATA_ROOT)
print("MODEL_ROOT   =", MODEL_ROOT)
print("OUTPUT_ROOT  =", OUTPUT_ROOT)
print("RAW_AUDIO    =", RAW_AUDIO_ROOT)
print("RAW_IMAGE    =", RAW_IMAGE_ROOT)
print("AUDIO_MANIFEST =", AUDIO_MANIFEST)
print("IMAGE_MANIFEST =", IMAGE_MANIFEST)


PROJECT_ROOT = /media/HwHiAiUser/aeedf028-4e56-4243-b4cd-5d61369ba3f9/qwen2/arabic
DATA_ROOT    = /media/HwHiAiUser/aeedf028-4e56-4243-b4cd-5d61369ba3f9/qwen2/arabic/data/WanJuanSiLu2O_ar
MODEL_ROOT   = /media/HwHiAiUser/aeedf028-4e56-4243-b4cd-5d61369ba3f9/qwen2/arabic/model/Qwen2-0.5B-Instruct
OUTPUT_ROOT  = /media/HwHiAiUser/aeedf028-4e56-4243-b4cd-5d61369ba3f9/qwen2/arabic/outputs
RAW_AUDIO    = /media/HwHiAiUser/aeedf028-4e56-4243-b4cd-5d61369ba3f9/qwen2/arabic/data/WanJuanSiLu2O_ar/OpenDataLab___WanJuanSiLu2O/raw/audio/ar
RAW_IMAGE    = /media/HwHiAiUser/aeedf028-4e56-4243-b4cd-5d61369ba3f9/qwen2/arabic/data/WanJuanSiLu2O_ar/OpenDataLab___WanJuanSiLu2O/raw/image/ar
AUDIO_MANIFEST = /media/HwHiAiUser/aeedf028-4e56-4243-b4cd-5d61369ba3f9/qwen2/arabic/outputs/audio_demo/extracted_manifest_multimodal.jsonl
IMAGE_MANIFEST = /media/HwHiAiUser/aeedf028-4e56-4243-b4cd-5d61369ba3f9/qwen2/arabic/outputs/image_demo/image_manifest_url.jsonl


In [2]:
import sys
import json
import subprocess
import shutil

# 假设 notebook 位于项目根目录内执行
PROJECT_ROOT = Path.cwd()

DATA_ROOT = PROJECT_ROOT / "data" / "WanJuanSiLu2O_ar"
MODEL_ROOT = PROJECT_ROOT / "model" / "Qwen2-0.5B-Instruct"
OUTPUT_ROOT = PROJECT_ROOT / "outputs"
AUDIO_OUT = OUTPUT_ROOT / "audio_demo"
IMAGE_OUT = OUTPUT_ROOT / "image_demo"

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
AUDIO_OUT.mkdir(parents=True, exist_ok=True)
IMAGE_OUT.mkdir(parents=True, exist_ok=True)

# 常用路径
RAW_AUDIO_ROOT = DATA_ROOT / "OpenDataLab___WanJuanSiLu2O" / "raw" / "audio" / "ar"
RAW_IMAGE_ROOT = DATA_ROOT / "OpenDataLab___WanJuanSiLu2O" / "raw" / "image" / "ar"

AUDIO_MANIFEST = AUDIO_OUT / "extracted_manifest_multimodal.jsonl"
IMAGE_MANIFEST = IMAGE_OUT / "image_manifest_url.jsonl"

print("PROJECT_ROOT =", PROJECT_ROOT)
print("DATA_ROOT    =", DATA_ROOT)
print("MODEL_ROOT   =", MODEL_ROOT)
print("OUTPUT_ROOT  =", OUTPUT_ROOT)
print("RAW_AUDIO    =", RAW_AUDIO_ROOT)
print("RAW_IMAGE    =", RAW_IMAGE_ROOT)
print("AUDIO_MANIFEST =", AUDIO_MANIFEST)
print("IMAGE_MANIFEST =", IMAGE_MANIFEST)


PROJECT_ROOT = /media/HwHiAiUser/aeedf028-4e56-4243-b4cd-5d61369ba3f9/qwen2/arabic
DATA_ROOT    = /media/HwHiAiUser/aeedf028-4e56-4243-b4cd-5d61369ba3f9/qwen2/arabic/data/WanJuanSiLu2O_ar
MODEL_ROOT   = /media/HwHiAiUser/aeedf028-4e56-4243-b4cd-5d61369ba3f9/qwen2/arabic/model/Qwen2-0.5B-Instruct
OUTPUT_ROOT  = /media/HwHiAiUser/aeedf028-4e56-4243-b4cd-5d61369ba3f9/qwen2/arabic/outputs
RAW_AUDIO    = /media/HwHiAiUser/aeedf028-4e56-4243-b4cd-5d61369ba3f9/qwen2/arabic/data/WanJuanSiLu2O_ar/OpenDataLab___WanJuanSiLu2O/raw/audio/ar
RAW_IMAGE    = /media/HwHiAiUser/aeedf028-4e56-4243-b4cd-5d61369ba3f9/qwen2/arabic/data/WanJuanSiLu2O_ar/OpenDataLab___WanJuanSiLu2O/raw/image/ar
AUDIO_MANIFEST = /media/HwHiAiUser/aeedf028-4e56-4243-b4cd-5d61369ba3f9/qwen2/arabic/outputs/audio_demo/extracted_manifest_multimodal.jsonl
IMAGE_MANIFEST = /media/HwHiAiUser/aeedf028-4e56-4243-b4cd-5d61369ba3f9/qwen2/arabic/outputs/image_demo/image_manifest_url.jsonl


## 2. 运行开关
把下面这些开关改成 `True`，就可以在 Notebook 里完成安装/下载/生成 manifest。

In [3]:

INSTALL_MISSING = False      # 是否自动安装缺失依赖
AUTO_OPENXLAB_LOGIN = False  # 是否使用环境变量自动登录 OpenXLab
DOWNLOAD_DATA = False        # 是否自动下载 WanJuanSiLu2O 阿语音频/图像数据
DOWNLOAD_MODEL = False       # 是否自动下载 Qwen2-0.5B-Instruct
BUILD_AUDIO_MANIFEST = True  # 是否生成音频 demo manifest
BUILD_IMAGE_MANIFEST = True  # 是否生成图像 URL manifest

# 资源数量控制
AUDIO_EXTRACT_LIMIT = 50
IMAGE_SAMPLE_LIMIT = 100

print({
    "INSTALL_MISSING": INSTALL_MISSING,
    "AUTO_OPENXLAB_LOGIN": AUTO_OPENXLAB_LOGIN,
    "DOWNLOAD_DATA": DOWNLOAD_DATA,
    "DOWNLOAD_MODEL": DOWNLOAD_MODEL,
    "BUILD_AUDIO_MANIFEST": BUILD_AUDIO_MANIFEST,
    "BUILD_IMAGE_MANIFEST": BUILD_IMAGE_MANIFEST,
    "AUDIO_EXTRACT_LIMIT": AUDIO_EXTRACT_LIMIT,
    "IMAGE_SAMPLE_LIMIT": IMAGE_SAMPLE_LIMIT,
})


{'INSTALL_MISSING': False, 'AUTO_OPENXLAB_LOGIN': False, 'DOWNLOAD_DATA': False, 'DOWNLOAD_MODEL': False, 'BUILD_AUDIO_MANIFEST': True, 'BUILD_IMAGE_MANIFEST': True, 'AUDIO_EXTRACT_LIMIT': 50, 'IMAGE_SAMPLE_LIMIT': 100}


## 3. 环境验证
检查 Python 版本、关键包、命令行工具和 NPU 工具是否可用。

In [4]:

import importlib

required_modules = [
    "mindspore",
    "mindnlp",
    "fastapi",
    "uvicorn",
]

print("Python =", sys.version)

for name in required_modules:
    try:
        mod = importlib.import_module(name)
        print(f"[OK] {name}: {getattr(mod, '__version__', 'unknown')}")
    except Exception as e:
        print(f"[MISSING] {name}: {e}")

for exe in ["openxlab", "git", "git-lfs", "npu-smi"]:
    path = shutil.which(exe)
    print(f"[CMD] {exe}: {path}")

if shutil.which("npu-smi"):
    try:
        result = subprocess.run(["npu-smi", "info"], capture_output=True, text=True, timeout=15)
        print(result.stdout[:2000])
    except Exception as e:
        print("npu-smi info failed:", e)


Python = 3.9.25 (main, Nov  3 2025, 22:33:42) 
[GCC 11.2.0]


/home/HwHiAiUser/.conda/envs/qwen/lib/python3.9/site-packages/numpy/core/getlimits.py:549: UserWarning: The value of the smallest subnormal for <class 'numpy.float64'> type is zero.
  setattr(self, word, getattr(machar, word).flat[0])
/home/HwHiAiUser/.conda/envs/qwen/lib/python3.9/site-packages/numpy/core/getlimits.py:89: UserWarning: The value of the smallest subnormal for <class 'numpy.float64'> type is zero.
  return self._float_to_str(self.smallest_subnormal)
/home/HwHiAiUser/.conda/envs/qwen/lib/python3.9/site-packages/numpy/core/getlimits.py:549: UserWarning: The value of the smallest subnormal for <class 'numpy.float32'> type is zero.
  setattr(self, word, getattr(machar, word).flat[0])
/home/HwHiAiUser/.conda/envs/qwen/lib/python3.9/site-packages/numpy/core/getlimits.py:89: UserWarning: The value of the smallest subnormal for <class 'numpy.float32'> type is zero.
  return self._float_to_str(self.smallest_subnormal)
[WARNING] ME(188000:246290333859872,MainProcess):2026-03-05-22

[OK] mindspore: 2.6.0


/home/HwHiAiUser/.conda/envs/qwen/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Building prefix dict from the default dictionary ...
Loading model from cache /tmp/jieba.cache
Loading model cost 1.698 seconds.
Prefix dict has been built successfully.
None of PyTorch, TensorFlow >= 2.0, or Flax have been found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.
None of PyTorch, TensorFlow >= 2.0, or Flax have been found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


[OK] mindnlp: unknown
[OK] fastapi: 0.128.8
[OK] uvicorn: 0.39.0
[CMD] openxlab: /home/HwHiAiUser/.conda/envs/qwen/bin/openxlab
[CMD] git: /usr/bin/git
[CMD] git-lfs: /usr/bin/git-lfs
[CMD] npu-smi: /usr/local/sbin/npu-smi
+--------------------------------------------------------------------------------------------------------+
| npu-smi 25.2.0                                   Version: 25.2.0                                       |
+-------------------------------+-----------------+------------------------------------------------------+
| NPU     Name                  | Health          | Power(W)     Temp(C)           Hugepages-Usage(page) |
| Chip    Device                | Bus-Id          | AICore(%)    Memory-Usage(MB)                        |
+===============================+=================+======================================================+
| 0       310B1                 | Alarm           | 0.0          47                15    / 15            |
| 0       0                 

## 4. 命令执行辅助函数
后续安装、下载都复用这个工具。

In [5]:

def run_cmd(cmd, check=True):
    print(">>", cmd)
    result = subprocess.run(cmd, shell=True, text=True)
    if check and result.returncode != 0:
        raise RuntimeError(f"Command failed: {cmd}")
    return result.returncode


## 5. 依赖安装（按需）
如果缺少 `openxlab`、`modelscope` 或其他包，可以打开 `INSTALL_MISSING=True` 再运行。

In [6]:

if INSTALL_MISSING:
    pkgs = [
        "openxlab",
        "modelscope",
        "huggingface_hub",
        "fastapi",
        "uvicorn",
    ]
    run_cmd(f"{sys.executable} -m pip install --upgrade " + " ".join(pkgs))
else:
    print("INSTALL_MISSING=False，跳过安装。")


INSTALL_MISSING=False，跳过安装。


## 6. OpenXLab 登录（可选）
如果你已在终端执行过 `openxlab login`，这一节可以跳过。  
如果想在 Notebook 自动登录，请先在当前环境中设置：

- `OPENXLAB_AK`
- `OPENXLAB_SK`


In [7]:

if AUTO_OPENXLAB_LOGIN:
    ak = os.environ.get("OPENXLAB_AK", "").strip()
    sk = os.environ.get("OPENXLAB_SK", "").strip()
    if not ak or not sk:
        raise ValueError("AUTO_OPENXLAB_LOGIN=True，但未设置 OPENXLAB_AK / OPENXLAB_SK")
    run_cmd(f'printf "%s\n%s\n" "{ak}" "{sk}" | openxlab login')
else:
    print("AUTO_OPENXLAB_LOGIN=False，跳过自动登录。")


AUTO_OPENXLAB_LOGIN=False，跳过自动登录。


## 7. 下载数据集（按需）
这里默认下载阿语的音频与图像子目录到 `data/WanJuanSiLu2O_ar/` 下。

使用的是 OpenXLab 的目录下载命令：

```bash
openxlab dataset download --dataset-repo OpenDataLab/WanJuanSiLu2O --source-path /raw/audio/ar --target-path data/WanJuanSiLu2O_ar/audio/
openxlab dataset download --dataset-repo OpenDataLab/WanJuanSiLu2O --source-path /raw/image/ar --target-path data/WanJuanSiLu2O_ar/image/
```

OpenXLab 的 CLI 用法可参考官方/社区说明。citeturn0search2turn0search12


In [ ]:
print(">>> 检查阿语多模态数据集是否已就绪...")

def check_data_ready():
    # 检查音频和图像根目录是否存在
    if not RAW_AUDIO_ROOT.exists() or not RAW_IMAGE_ROOT.exists():
        return False
    # 检查里面是否有实际文件（避免只建了个空文件夹）
    audio_files = list(RAW_AUDIO_ROOT.rglob('*'))
    image_files = list(RAW_IMAGE_ROOT.rglob('*'))
    if len(audio_files) < 5 or len(image_files) < 5:
        return False
    return True

if check_data_ready():
    print("✅ 数据集已完整存在，跳过下载！")
    print(f"👉 音频数据路径: {RAW_AUDIO_ROOT}")
    print(f"👉 图像数据路径: {RAW_IMAGE_ROOT}")
else:
    print("⚠️ 未检测到完整数据集，开始从 OpenDataLab 自动下载 (根据网络情况可能需要一段时间)...")
    DATA_ROOT.mkdir(parents=True, exist_ok=True)
    
    # 下载音频数据
    run_cmd(
        f'openxlab dataset download '
        f'--dataset-repo OpenDataLab/WanJuanSiLu2O '
        f'--source-path /raw/audio/ar '
        f'--target-path "{DATA_ROOT}"'
    )
    # 下载图像数据
    run_cmd(
        f'openxlab dataset download '
        f'--dataset-repo OpenDataLab/WanJuanSiLu2O '
        f'--source-path /raw/image/ar '
        f'--target-path "{DATA_ROOT}"'
    )
    print("✅ 数据集自动下载完成！")

DOWNLOAD_DATA=False，跳过数据下载。


## 8. 下载模型（按需）
这里优先用 `modelscope` 的 `snapshot_download` 下载 `Qwen2-0.5B-Instruct` 到相对路径 `model/Qwen2-0.5B-Instruct/`。


In [ ]:
print(">>> 检查 Qwen2-0.5B-Instruct 模型是否已就绪...")

def check_model_ready():
    # 检查模型目录是否存在
    if not MODEL_ROOT.exists():
        return False
    # 检查核心配置文件和分词器文件是否存在
    if not (MODEL_ROOT / "config.json").exists() or not (MODEL_ROOT / "vocab.json").exists():
        return False
    return True

if check_model_ready():
    print("✅ 模型文件已完好存在，跳过下载！")
    print(f"👉 模型路径: {MODEL_ROOT}")
else:
    print("⚠️ 未检测到模型或模型文件缺失，开始从 ModelScope 下载 Qwen2-0.5B-Instruct...")
    try:
        from modelscope import snapshot_download
        MODEL_ROOT.parent.mkdir(parents=True, exist_ok=True)
        
        # 自动下载模型到指定目录
        snapshot_download(
            "Qwen/Qwen2-0.5B-Instruct",
            local_dir=str(MODEL_ROOT),
            local_dir_use_symlinks=False
        )
        print(f"✅ 模型自动下载并校验完成！路径: {MODEL_ROOT}")
    except ImportError:
        print("❌ 缺少 modelscope 库。请先执行: pip install modelscope")
    except Exception as e:
        print(f"❌ 模型下载失败，请检查网络: {e}")

DOWNLOAD_MODEL=False，跳过模型下载。


## 9. 检查关键目录
确认模型目录、数据目录是否已经准备好。

In [10]:

print("MODEL_ROOT exists:", MODEL_ROOT.exists())
print("RAW_AUDIO_ROOT exists:", RAW_AUDIO_ROOT.exists())
print("RAW_IMAGE_ROOT exists:", RAW_IMAGE_ROOT.exists())

if MODEL_ROOT.exists():
    print("MODEL_ROOT files:", sorted([p.name for p in MODEL_ROOT.iterdir()])[:20])

if RAW_AUDIO_ROOT.exists():
    print("RAW_AUDIO sample files:", [str(p) for p in list(RAW_AUDIO_ROOT.rglob('*'))[:10]])

if RAW_IMAGE_ROOT.exists():
    print("RAW_IMAGE sample files:", [str(p) for p in list(RAW_IMAGE_ROOT.rglob('*'))[:10]])


MODEL_ROOT exists: True
RAW_AUDIO_ROOT exists: True
RAW_IMAGE_ROOT exists: True
MODEL_ROOT files: ['.git', '.gitattributes', 'LICENSE', 'README.md', 'config.json', 'configuration.json', 'generation_config.json', 'kernel_meta', 'merges.txt', 'model.safetensors', 'offload', 'tokenizer.json', 'tokenizer_config.json', 'tokenizer_config.json.lock', 'vocab.json']
RAW_AUDIO sample files: ['/media/HwHiAiUser/aeedf028-4e56-4243-b4cd-5d61369ba3f9/qwen2/arabic/data/WanJuanSiLu2O_ar/OpenDataLab___WanJuanSiLu2O/raw/audio/ar/audio', '/media/HwHiAiUser/aeedf028-4e56-4243-b4cd-5d61369ba3f9/qwen2/arabic/data/WanJuanSiLu2O_ar/OpenDataLab___WanJuanSiLu2O/raw/audio/ar/_extracted_demo', '/media/HwHiAiUser/aeedf028-4e56-4243-b4cd-5d61369ba3f9/qwen2/arabic/data/WanJuanSiLu2O_ar/OpenDataLab___WanJuanSiLu2O/raw/audio/ar/json', '/media/HwHiAiUser/aeedf028-4e56-4243-b4cd-5d61369ba3f9/qwen2/arabic/data/WanJuanSiLu2O_ar/OpenDataLab___WanJuanSiLu2O/raw/audio/ar/audio/ar_part4.zip', '/media/HwHiAiUser/aeedf028-4e56-

## 10. 生成 Audio demo manifest
从阿语音频数据中抽取少量 wav 与转写文本，生成网页与测试用的 `extracted_manifest_multimodal.jsonl`。

In [11]:

import gzip
import zipfile

def iter_jsonl_records(jsonl_fp):
    open_fn = gzip.open if str(jsonl_fp).endswith(".gz") else open
    with open_fn(jsonl_fp, "rt", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                yield json.loads(line)
            except Exception:
                continue

def build_audio_manifest():
    jsonl_candidates = sorted(RAW_AUDIO_ROOT.rglob("*.jsonl")) + sorted(RAW_AUDIO_ROOT.rglob("*.jsonl.gz"))
    zip_files = sorted(RAW_AUDIO_ROOT.rglob("*.zip"))
    extract_dir = AUDIO_OUT / "_extracted_wavs"
    extract_dir.mkdir(parents=True, exist_ok=True)

    if not jsonl_candidates:
        print("No audio jsonl/jsonl.gz found under:", RAW_AUDIO_ROOT)
        return

    print("Found audio jsonl files:", len(jsonl_candidates))
    print("Found audio zip files:", len(zip_files))

    # 建一个 basename -> zip path 的索引
    zip_index = {}
    for z in zip_files:
        zip_index[z.name] = z

    written = 0
    with open(AUDIO_MANIFEST, "w", encoding="utf-8") as fout:
        for jf in jsonl_candidates:
            for item in iter_jsonl_records(jf):
                if written >= AUDIO_EXTRACT_LIMIT:
                    break

                audio = item.get("audio", {})
                text = item.get("text", {})
                rel_path = audio.get("path", "")
                transcript = text.get("content", "")

                if not rel_path or not transcript:
                    continue

                wav_name = Path(rel_path).name
                wav_out = extract_dir / wav_name

                # 尝试从 zip 中提取
                extracted = False
                for zf in zip_files:
                    try:
                        with zipfile.ZipFile(zf, "r") as z:
                            if wav_name in z.namelist():
                                if not wav_out.exists():
                                    with z.open(wav_name) as src, open(wav_out, "wb") as dst:
                                        dst.write(src.read())
                                extracted = True
                                break
                    except Exception:
                        continue

                if not extracted:
                    continue

                out_obj = {
                    "id": item.get("id", f"audio_{written}"),
                    "wav_path": str(wav_out),
                    "wav_name": wav_name,
                    "duration": audio.get("duration", ""),
                    "sample_rate": audio.get("sample_rate", ""),
                    "channels": audio.get("channels", ""),
                    "snr": audio.get("snr", ""),
                    "audio_format": audio.get("format", ""),
                    "transcript": transcript,
                }
                fout.write(json.dumps(out_obj, ensure_ascii=False) + "\n")
                written += 1

            if written >= AUDIO_EXTRACT_LIMIT:
                break

    print("Audio manifest saved to:", AUDIO_MANIFEST)
    print("Audio samples written:", written)

if BUILD_AUDIO_MANIFEST:
    build_audio_manifest()
else:
    print("BUILD_AUDIO_MANIFEST=False，跳过生成。")


Found audio jsonl files: 3
Found audio zip files: 9
Audio manifest saved to: /media/HwHiAiUser/aeedf028-4e56-4243-b4cd-5d61369ba3f9/qwen2/arabic/outputs/audio_demo/extracted_manifest_multimodal.jsonl
Audio samples written: 50


## 11. 生成 Image URL manifest
从阿语图像标注中提取 `image.path` 与 `captions.content`，生成 `image_manifest_url.jsonl`。

In [12]:

def load_jsonl_file(fp):
    items = []
    if not fp.exists():
        return items
    with open(fp, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                items.append(json.loads(line))
            except Exception:
                continue
    return items

def pick_image_url(item):
    img = item.get("image")
    if isinstance(img, dict):
        p = img.get("path")
        if isinstance(p, str) and p.startswith(("http://", "https://")):
            return p.strip()

    for k in ["image_path", "url", "path"]:
        v = item.get(k)
        if isinstance(v, str) and v.startswith(("http://", "https://")):
            return v.strip()
    return None

def pick_image_text(item):
    caps = item.get("captions")
    if isinstance(caps, dict):
        v = caps.get("content")
        if isinstance(v, str) and v.strip():
            return v.strip()

    if isinstance(caps, str) and caps.strip():
        return caps.strip()

    if isinstance(caps, list):
        texts = []
        for x in caps:
            if isinstance(x, str) and x.strip():
                texts.append(x.strip())
            elif isinstance(x, dict):
                for kk in ["content", "text", "caption", "value"]:
                    v = x.get(kk)
                    if isinstance(v, str) and v.strip():
                        texts.append(v.strip())
                        break
        if texts:
            return "\n".join(texts[:3])

    return ""

def build_image_manifest():
    candidates = [
        RAW_IMAGE_ROOT / "ar_image_caption.jsonl",
        RAW_IMAGE_ROOT / "ar_image_text_pair.jsonl",
    ]

    all_items = []
    for fp in candidates:
        data = load_jsonl_file(fp)
        print(f"loaded {len(data)} from {fp}")
        all_items.extend(data)

    written = 0
    with open(IMAGE_MANIFEST, "w", encoding="utf-8") as fout:
        for item in all_items:
            if written >= IMAGE_SAMPLE_LIMIT:
                break

            image_url = pick_image_url(item)
            text = pick_image_text(item)

            if not image_url or not text:
                continue

            out_obj = {
                "id": item.get("img_id", f"img_{written}"),
                "image_url": image_url,
                "image_name": image_url.split("/")[-1],
                "text": text,
            }
            fout.write(json.dumps(out_obj, ensure_ascii=False) + "\n")
            written += 1

    print("Image manifest saved to:", IMAGE_MANIFEST)
    print("Image samples written:", written)

if BUILD_IMAGE_MANIFEST:
    build_image_manifest()
else:
    print("BUILD_IMAGE_MANIFEST=False，跳过生成。")


loaded 24124 from /media/HwHiAiUser/aeedf028-4e56-4243-b4cd-5d61369ba3f9/qwen2/arabic/data/WanJuanSiLu2O_ar/OpenDataLab___WanJuanSiLu2O/raw/image/ar/ar_image_caption.jsonl
loaded 198198 from /media/HwHiAiUser/aeedf028-4e56-4243-b4cd-5d61369ba3f9/qwen2/arabic/data/WanJuanSiLu2O_ar/OpenDataLab___WanJuanSiLu2O/raw/image/ar/ar_image_text_pair.jsonl
Image manifest saved to: /media/HwHiAiUser/aeedf028-4e56-4243-b4cd-5d61369ba3f9/qwen2/arabic/outputs/image_demo/image_manifest_url.jsonl
Image samples written: 100


## 12. 检查生成后的 manifest
确认音频与图像 manifest 都已经生成。

In [13]:

for fp in [AUDIO_MANIFEST, IMAGE_MANIFEST]:
    print("=" * 80)
    print(fp)
    if fp.exists():
        print("exists = True")
        with open(fp, "r", encoding="utf-8") as f:
            for i, line in enumerate(f):
                print(line[:300])
                if i >= 1:
                    break
    else:
        print("exists = False")


/media/HwHiAiUser/aeedf028-4e56-4243-b4cd-5d61369ba3f9/qwen2/arabic/outputs/audio_demo/extracted_manifest_multimodal.jsonl
exists = True
{"id": "e734425d-c9ca-4502-a8f5-16c4710ae531", "wav_path": "/media/HwHiAiUser/aeedf028-4e56-4243-b4cd-5d61369ba3f9/qwen2/arabic/outputs/audio_demo/_extracted_wavs/--lLrhhJ3QA_segment3.wav", "wav_name": "--lLrhhJ3QA_segment3.wav", "duration": "27.68 s", "sample_rate": "16000 Hz", "channels": "1", "sn
{"id": "06bd627e-a1c5-40d8-bf94-7355df7ec0a3", "wav_path": "/media/HwHiAiUser/aeedf028-4e56-4243-b4cd-5d61369ba3f9/qwen2/arabic/outputs/audio_demo/_extracted_wavs/-0R1I26YwAE_segment6.wav", "wav_name": "-0R1I26YwAE_segment6.wav", "duration": "1.75 s", "sample_rate": "16000 Hz", "channels": "1", "snr
/media/HwHiAiUser/aeedf028-4e56-4243-b4cd-5d61369ba3f9/qwen2/arabic/outputs/image_demo/image_manifest_url.jsonl
exists = True
{"id": "00347187fcf81a58b7184f046d9d8c23253eb95e22f47834ae5746768bc3ebbc", "image_url": "https://www.okaz.com.sa/uploads/images/2017/12

## 13. 导入推理与 Web 应用依赖
这一节开始进入模型与 Web 应用主体代码。

In [14]:

import json
import time
import threading
from pathlib import Path

import mindspore as ms
from mindspore import context
from mindnlp.transformers import AutoTokenizer, AutoModelForCausalLM

from fastapi import FastAPI, Query
from fastapi.responses import HTMLResponse, FileResponse, JSONResponse
import uvicorn

infer_lock = threading.Lock()

MODEL_PATH = str(MODEL_ROOT)
AUDIO_MANIFEST = str(AUDIO_MANIFEST)
IMAGE_MANIFEST = str(IMAGE_MANIFEST)


## 2. 初始化 MindSpore 运行环境
这里设置 Ascend 设备、PyNative 模式和同步执行，优先保证板端推理稳定性。

In [15]:
context.set_context(
    mode=context.PYNATIVE_MODE,
    device_target="Ascend",
    jit_level="O0",
    pynative_synchronize=True
)

print(">>> MindSpore context initialized.")

[WARNING] ME(188000:246290333859872,MainProcess):2026-03-05-22:24:38.855.565 [mindspore/context.py:1402] For 'context.set_context', the parameter 'device_target' will be deprecated and removed in a future version. Please use the api mindspore.set_device() instead.
[WARNING] ME(188000:246290333859872,MainProcess):2026-03-05-22:24:38.860.218 [mindspore/context.py:1402] For 'context.set_context', the parameter 'pynative_synchronize' will be deprecated and removed in a future version. Please use the api mindspore.runtime.launch_blocking() instead.


>>> MindSpore context initialized.


## 3. 定义 manifest 读取函数
音频与图像样本都通过 jsonl manifest 管理，因此先写一个通用读取函数。

In [16]:
def load_manifest(fp):
    items = []
    p = Path(fp)
    if not p.exists():
        print(f">>> manifest not found: {fp}")
        return items

    with open(p, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                items.append(json.loads(line))
            except Exception:
                continue
    return items

## 4. 加载音频与图像样本索引
这一部分读取两份 manifest，并确认样本数是否正常。

In [17]:
audio_samples = load_manifest(AUDIO_MANIFEST)
image_samples = load_manifest(IMAGE_MANIFEST)

print(f">>> audio samples: {len(audio_samples)}")
print(f">>> image samples: {len(image_samples)}")

>>> audio samples: 50
>>> image samples: 100


## 5. 加载 tokenizer 与模型
这里加载 Qwen2-0.5B-Instruct，并关闭采样参数，适合网页问答场景。

In [18]:
print(">>> Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)

print(">>> Loading model...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    ms_dtype=ms.float16
)
model.set_train(False)

if hasattr(model, "generation_config") and model.generation_config is not None:
    model.generation_config.do_sample = False
    model.generation_config.temperature = None
    model.generation_config.top_p = None
    model.generation_config.top_k = None

print(">>> Model loaded.")

>>> Loading tokenizer...
>>> Loading model...


Qwen2ForCausalLM has generative capabilities, as `prepare_inputs_for_generation` is explicitly overwritten. However, it doesn't directly inherit from `GenerationMixin`.`PreTrainedModel` will NOT inherit from `GenerationMixin`, and this model will lose the ability to call `generate` and other related functions.
  - If you are the owner of the model architecture code, please modify your model class such that it inherits from `GenerationMixin` (after `PreTrainedModel`, otherwise you'll get an exception).
  - If you are not the owner of the model architecture class, please contact the model code owner to update it.
Sliding Window Attention is enabled but not implemented for `eager`; unexpected results may be encountered.


>>> Model loaded.


## 6. 构造 Audio Prompt
这个函数把音频元信息、转写文本和用户问题拼成阿拉伯语 prompt。

In [19]:
def build_audio_prompt(sample, question):
    transcript = sample.get("transcript", "")
    duration = sample.get("duration", "")
    sample_rate = sample.get("sample_rate", "")
    channels = sample.get("channels", "")
    snr = sample.get("snr", "")
    wav_name = sample.get("wav_name", "")
    audio_format = sample.get("audio_format", "")

    prompt = f"""أنت مساعد لفهم الصوت العربي والإجابة عن الأسئلة.

فيما يلي معلومات عن عينة صوتية عربية:

اسم الملف: {wav_name}
صيغة الصوت: {audio_format}
مدة الصوت: {duration}
معدل أخذ العينات: {sample_rate}
عدد القنوات: {channels}
نسبة الإشارة إلى الضجيج: {snr}

النص المفرغ:
{transcript}

السؤال:
{question}

أجب بالعربية بإيجاز ووضوح."""
    return prompt

## 7. 构造 Image Prompt
这个函数把图片 URL、图片说明文本和用户问题拼成阿拉伯语 prompt。

In [20]:
def build_image_prompt(sample, question):
    image_name = sample.get("image_name", "")
    image_url = sample.get("image_url", "")
    text = sample.get("text", "")

    prompt = f"""أنت مساعد لفهم الصور والنصوص العربية والإجابة عن الأسئلة.

فيما يلي معلومات عن عينة صورة:

اسم 图片: {image_name}
رابط 图片: {image_url}

图片说明文本:
{text}

السؤال:
{question}

أجب بالعربية بإيجاز ووضوح."""
    return prompt

## 8. 定义统一推理函数
网页端音频问答与图像问答都会复用这个生成函数。

In [21]:
def infer_with_prompt(prompt, max_new_tokens=32):
    inputs = tokenizer(prompt, return_tensors="ms")

    output_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        use_cache=False,
        pad_token_id=tokenizer.eos_token_id if tokenizer.eos_token_id is not None else None,
        eos_token_id=tokenizer.eos_token_id if tokenizer.eos_token_id is not None else None,
    )

    try:
        input_len = inputs["input_ids"].shape[1]
        gen_ids = output_ids[:, input_len:]
    except Exception:
        gen_ids = output_ids

    answer = tokenizer.batch_decode(
        gen_ids,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=True
    )[0]

    return answer.strip()

## 9. Audio QA 单样本测试
在启动 Web 服务前，建议先做一轮音频样本测试，确认模型与数据链路正常。

In [22]:
if audio_samples:
    audio_idx = 0
    audio_question = "ما الموضوع الرئيسي لهذا المقطع الصوتي؟"
    audio_sample = audio_samples[audio_idx]

    print("wav_path:", audio_sample.get("wav_path", ""))
    print("transcript:", audio_sample.get("transcript", "")[:200])

    t0 = time.time()
    with infer_lock:
        audio_answer = infer_with_prompt(build_audio_prompt(audio_sample, audio_question))
    t1 = time.time()

    print("\n[Audio Answer]")
    print(audio_answer)
    print(f"\n[infer_sec] {{t1 - t0:.3f}}")
else:
    print("No audio samples found.")

wav_path: /media/HwHiAiUser/aeedf028-4e56-4243-b4cd-5d61369ba3f9/qwen2/arabic/outputs/audio_demo/_extracted_wavs/--lLrhhJ3QA_segment3.wav
transcript: نعيش بيهم في يوم هجرة النبي صلى الله عليه وسلم نعيش بيهم في بداية سنة هقوية جديدة. الثلث معاني دول المعنى الأولاني، إن من قلب الشدة يولد الأمل، يا جماعة النبي كان مهاجر من مكة للمدينة، الأمور صعبة جد 


2026-03-05 22:26:47.100884: E external/org_tensorflow/tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute is_closed which is not in the op definition: Op<name=Range; signature=start:Tidx, limit:Tidx, delta:Tidx -> output:Tidx; attr=Tidx:type,default=DT_INT32,allowed=[DT_BFLOAT16, DT_HALF, DT_FLOAT, DT_DOUBLE, DT_INT8, DT_INT16, DT_INT32, DT_INT64, DT_UINT16, DT_UINT32]> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node Range4}}



[Audio Answer]
"الموضوع الرئيسي لهذا المقطع الصوتي هو تفسير القرآن الكريم."

[infer_sec] {t1 - t0:.3f}


## 10. Image QA 单样本测试
这一部分验证图像分支是否能正常读取 caption 文本并完成问答。

In [23]:
if image_samples:
    image_idx = 0
    image_question = "ما الذي تصفه هذه الصورة بشكل رئيسي؟"
    image_sample = image_samples[image_idx]

    print("image_url:", image_sample.get("image_url", ""))
    print("text:", image_sample.get("text", "")[:300])

    t0 = time.time()
    with infer_lock:
        image_answer = infer_with_prompt(build_image_prompt(image_sample, image_question))
    t1 = time.time()

    print("\n[Image Answer]")
    print(image_answer)
    print(f"\n[infer_sec] {{t1 - t0:.3f}}")
else:
    print("No image samples found.")

image_url: https://www.okaz.com.sa/uploads/images/2017/12/27/647923.jpeg
text: تُظهر هذه الصورة شخصًا يرتدي بدلة واقية بيضاء ويقوم بتنظيف بيئة داخلية. يدير الرجل ظهره للكاميرا ويستخدم أداة تنظيف زرقاء تشبه مسدس الماء عالي الضغط لتنظيف الأرضية. في الخلفية، يمكننا رؤية رف مليء بمجموعة متنوعة من العناصر، بما في ذلك أغذية الحيوانات الأليفة وفضلات القطط ومستلزمات الحيوانات الأليفة 

[Image Answer]
الإجابة:
هذه الصورة تصف الشخص يرتدي بدلة واقية بيضاء ويقوم بتنظيف بيئة داخلية. ي

[infer_sec] {t1 - t0:.3f}


## 11. 创建 FastAPI 应用对象
下面开始逐步拼装网页应用。

In [24]:
app = FastAPI()

## 12. 定义首页 `/`
首页提供两个入口：音频问答与图像问答。

In [25]:
@app.get("/", response_class=HTMLResponse)
def home():
    html = """
    <html>
    <head>
        <meta charset="utf-8">
        <title>万卷·丝路 - 阿拉伯语多模态问答系统</title>
        <style>
            body { font-family: 'Microsoft YaHei', Arial, sans-serif; max-width: 900px; margin: 24px auto; line-height: 1.8; }
            a { font-size: 18px; color: #0366d6; text-decoration: none; }
            a:hover { text-decoration: underline; }
            .badge { background: #f0f6fc; border: 1px solid #d0d7de; padding: 4px 8px; border-radius: 6px; font-size: 14px; }
        </style>
    </head>
    <body>
        <h2>🌍 万卷·丝路 - 阿拉伯语多模态问答系统</h2>
        <p>
            <span class="badge">硬件: OrangePi AIpro (Ascend 310B)</span>
            <span class="badge">模型: Qwen2-0.5B-Instruct</span>
            <span class="badge">框架: MindSpore + MindNLP</span>
        </p>
        <p>请选择你要测试的多模态场景：</p>
        <ul>
            <li><a href="/audio">🎧 音频问答测试 (Audio QA)</a></li>
            <li><a href="/image">🖼️ 图像问答测试 (Image QA)</a></li>
        </ul>
    </body>
    </html>
    """
    return HTMLResponse(html)

## 13. 定义音频页面 `/audio`
这个页面负责展示音频播放器、音频元信息、转写文本、问题输入框与模型回答。

In [26]:
@app.get("/audio", response_class=HTMLResponse)
def audio_page():
    options = []
    for i, s in enumerate(audio_samples):
        title = f"{i} | {s.get('wav_name', '')} | {s.get('duration', '')}"
        options.append(f"<option value='{i}'>{title}</option>")
    options_html = "\n".join(options)

    html = """
    <html>
    <head>
        <meta charset="utf-8">
        <title>音频问答测试 - 阿拉伯语</title>
        <style>
            body { font-family: 'Microsoft YaHei', Arial, sans-serif; max-width: 1100px; margin: 20px auto; line-height: 1.8; }
            .row { display: flex; gap: 20px; } .col { flex: 1; }
            /* 中文界面用正常方向，阿语内容专门使用 rtl 方向 */
            .arabic-text { direction: rtl; text-align: right; font-size: 16px; font-family: Arial, sans-serif; }
            textarea { width: 100%; min-height: 100px; padding: 8px; }
            pre { background: #f6f6f6; padding: 12px; border-radius: 8px; white-space: pre-wrap; font-size: 14px; }
            button { padding: 8px 16px; font-size: 15px; cursor: pointer; background-color: #0969da; color: white; border: none; border-radius: 6px; }
            button:hover { background-color: #0353a4; }
            select { min-width: 520px; padding: 6px; font-size: 14px; }
            a { color: #0969da; text-decoration: none; }
        </style>
    </head>
    <body>
        <h2>🎧 音频问答测试 (Audio QA)</h2>
        <p><a href="/">← 返回首页</a></p>

        <label><b>选择数据样本：</b></label><br/>
        <select id="idx">__OPTIONS_HTML__</select>
        <button onclick="loadSample()">加载样本</button>

        <div class="row" style="margin-top:20px;">
            <div class="col">
                <h3>🔊 播放音频</h3>
                <audio id="player" controls style="width:100%;"></audio>
                <h3>📊 音频元数据 (Metadata)</h3>
                <pre id="meta"></pre>
            </div>
            <div class="col">
                <h3>📝 阿拉伯语转写文本 (Transcript)</h3>
                <pre id="transcript" class="arabic-text"></pre>
            </div>
        </div>

        <h3>❓ 请输入问题 (支持阿拉伯语)</h3>
        <textarea id="question" class="arabic-text">ما الموضوع الرئيسي لهذا المقطع الصوتي؟</textarea><br/>
        <button onclick="askQuestion()" style="margin-top: 10px;">🚀 开始推理</button>

        <h3>🤖 模型回答 (Answer)</h3>
        <pre id="answer" class="arabic-text"></pre>

        <script>
        function getSafeIdx() {
            const sel = document.getElementById("idx");
            if (!sel || !sel.value) return 0;
            return parseInt(sel.value, 10) || 0;
        }

        async function loadSample() {
            const idx = getSafeIdx();
            const res = await fetch("/audio_sample?idx=" + idx);
            const data = await res.json();

            if (data.error) {
                document.getElementById("meta").textContent = data.error;
                return;
            }

            document.getElementById("player").src = "/audio_wav?idx=" + idx;
            document.getElementById("transcript").textContent = data.transcript || "无转写文本";
            
            // 将元数据标签中文化
            document.getElementById("meta").textContent =
                "文件名: " + (data.wav_name || "未知") + "\\n" +
                "音频时长: " + (data.duration || "未知") + "\\n" +
                "采样率: " + (data.sample_rate || "未知") + "\\n" +
                "声道数: " + (data.channels || "未知") + "\\n" +
                "信噪比(SNR): " + (data.snr || "未知") + "\\n" +
                "音频格式: " + (data.audio_format || "未知");
            document.getElementById("answer").textContent = "等待提问...";
        }

        async function askQuestion() {
            const idx = getSafeIdx();
            const q = document.getElementById("question").value || "";
            document.getElementById("answer").textContent = "推理中，请稍候 (Running Inference)...";

            const res = await fetch("/audio_qa?idx=" + idx + "&q=" + encodeURIComponent(q));
            const data = await res.json();

            if (data.error) {
                document.getElementById("answer").textContent = "发生错误: " + data.error;
            } else {
                document.getElementById("answer").textContent =
                    (data.answer || "") + "\\n\\n[推理耗时] " + Number(data.infer_sec || 0).toFixed(3) + " 秒";
            }
        }

        window.onload = loadSample;
        </script>
    </body>
    </html>
    """
    html = html.replace("__OPTIONS_HTML__", options_html)
    return HTMLResponse(html)

## 14. 定义音频相关 API
包括：音频样本信息、音频文件流、音频问答接口。

In [27]:
@app.get("/audio_sample")
def get_audio_sample(idx: int = Query(...)):
    if idx < 0 or idx >= len(audio_samples):
        return JSONResponse({"error": "idx out of range"}, status_code=400)

    s = audio_samples[idx]
    return {
        "wav_name": s.get("wav_name", ""),
        "duration": s.get("duration", ""),
        "sample_rate": s.get("sample_rate", ""),
        "channels": s.get("channels", ""),
        "snr": s.get("snr", ""),
        "audio_format": s.get("audio_format", ""),
        "transcript": s.get("transcript", "")
    }


@app.get("/audio_wav")
def get_audio_wav(idx: int = Query(...)):
    if idx < 0 or idx >= len(audio_samples):
        return JSONResponse({"error": "idx out of range"}, status_code=400)

    wav_path = Path(audio_samples[idx]["wav_path"])
    if not wav_path.exists():
        return JSONResponse({"error": f"wav not found: {wav_path}"}, status_code=404)

    return FileResponse(str(wav_path), media_type="audio/wav", filename=wav_path.name)


@app.get("/audio_qa")
def audio_qa(idx: int = Query(...), q: str = Query("")):
    if idx < 0 or idx >= len(audio_samples):
        return JSONResponse({"error": "idx out of range"}, status_code=400)

    if not q.strip():
        return {"error": "الرجاء إدخال السؤال."}

    sample = audio_samples[idx]
    t0 = time.time()
    try:
        with infer_lock:
            answer = infer_with_prompt(build_audio_prompt(sample, q.strip()))
        t1 = time.time()
        return {
            "answer": answer,
            "infer_sec": t1 - t0
        }
    except Exception as e:
        return {"error": repr(e)}

## 15. 定义图像页面 `/image`
这个页面展示图片、图片 URL、caption 文本、问题输入框和模型回答。

In [28]:
@app.get("/image", response_class=HTMLResponse)
def image_page():
    options = [f"<option value='{i}'>{i} | {s.get('image_name', '')}</option>" for i, s in enumerate(image_samples)]
    options_html = "\n".join(options)

    html = """
    <html>
    <head>
        <meta charset="utf-8">
        <title>图像问答测试 - 阿拉伯语</title>
        <style>
            body { font-family: 'Microsoft YaHei', Arial, sans-serif; max-width: 1100px; margin: 20px auto; line-height: 1.8; }
            .row { display: flex; gap: 20px; } .col { flex: 1; }
            .arabic-text { direction: rtl; text-align: right; font-size: 16px; font-family: Arial, sans-serif; }
            textarea { width: 100%; min-height: 100px; padding: 8px; }
            pre { background: #f6f6f6; padding: 12px; border-radius: 8px; white-space: pre-wrap; font-size: 14px; word-wrap: break-word; }
            button { padding: 8px 16px; font-size: 15px; cursor: pointer; background-color: #0969da; color: white; border: none; border-radius: 6px; }
            button:hover { background-color: #0353a4; }
            select { min-width: 520px; padding: 6px; font-size: 14px; }
            a { color: #0969da; text-decoration: none; }
            img { max-width: 100%; max-height: 520px; border: 1px solid #ddd; border-radius: 8px; background-color: #f9f9f9; }
        </style>
    </head>
    <body>
        <h2>🖼️ 图像问答测试 (Image QA)</h2>
        <p><a href="/">← 返回首页</a></p>

        <label><b>选择数据样本：</b></label><br/>
        <select id="idx" onchange="loadSample()">__OPTIONS_HTML__</select>
        <button onclick="loadSample()">加载样本</button>

        <div class="row" style="margin-top:20px;">
            <div class="col">
                <h3>🖼️ 显示图像</h3>
                <img id="img" alt="正在加载图像..." onerror="this.onerror=null; this.src='data:image/svg+xml;charset=utf-8,%3Csvg xmlns%3D\\'http%3A%2F%2Fwww.w3.org%2F2000%2Fsvg\\' width%3D\\'400\\' height%3D\\'300\\'%3E%3Crect width%3D\\'400\\' height%3D\\'300\\' fill%3D\\'%23f0f0f0\\'%2F%3E%3Ctext x%3D\\'50%25\\' y%3D\\'50%25\\' dominant-baseline%3D\\'middle\\' text-anchor%3D\\'middle\\' font-size%3D\\'18\\' fill%3D\\'%23666\\'%3E%E2%9A%A0%EF%B8%8F %E5%8E%9F%E5%9B%BE%E9%93%BE%E6%8E%A5%E5%B7%B2%E5%9C%A8%E4%BA%92%E8%81%94%E7%BD%91%E4%B8%8A%E5%A4%B1%E6%95%88%3C%2Ftext%3E%3C%2Fsvg%3E';" />
                <h3>🔗 图像原始链接</h3>
                <pre id="imgurl"></pre>
            </div>
            <div class="col">
                <h3>📝 图像关联文本 (Caption)</h3>
                <pre id="text" class="arabic-text"></pre>
            </div>
        </div>

        <h3>❓ 请输入问题 (支持阿拉伯语)</h3>
        <textarea id="question" class="arabic-text">ما الذي تصفه هذه الصورة بشكل رئيسي؟</textarea><br/>
        <button onclick="askQuestion()" style="margin-top: 10px;">🚀 开始推理</button>

        <h3>🤖 模型回答 (Answer)</h3>
        <pre id="answer" class="arabic-text"></pre>

        <script>
        function getSafeIdx() {
            const sel = document.getElementById("idx");
            if (!sel || !sel.value) return 0;
            return parseInt(sel.value, 10) || 0;
        }

        async function loadSample() {
            const idx = getSafeIdx();
            const res = await fetch("/image_sample?idx=" + idx);
            const data = await res.json();

            if (data.error) {
                document.getElementById("text").textContent = data.error;
                return;
            }

            // 【关键修改】直接使用图片的真实外网 URL 交给浏览器加载！
            document.getElementById("img").src = ""; 
            if (data.image_url) {
                document.getElementById("img").src = data.image_url;
            }
            
            document.getElementById("imgurl").textContent = data.image_url || "无链接";
            document.getElementById("text").textContent = data.text || "无说明文本";
            document.getElementById("answer").textContent = "等待提问...";
        }

        async function askQuestion() {
            const idx = getSafeIdx();
            const q = document.getElementById("question").value || "";
            document.getElementById("answer").textContent = "推理中，请稍候 (Running Inference)...";

            const res = await fetch(`/image_qa?idx=${idx}&q=${encodeURIComponent(q)}`);
            const data = await res.json();

            if (data.error) {
                document.getElementById("answer").textContent = "发生错误: " + data.error;
            } else {
                document.getElementById("answer").textContent = 
                    data.answer + "\\n\\n[推理耗时] " + Number(data.infer_sec || 0).toFixed(3) + " 秒";
            }
        }

        window.onload = loadSample;
        </script>
    </body>
    </html>
    """
    html = html.replace("__OPTIONS_HTML__", options_html)
    return HTMLResponse(html)

## 16. 定义图像相关 API
包括：图像样本信息与图像问答接口。

In [29]:
@app.get("/image_sample")
def get_image_sample(idx: int = Query(...)):
    if idx < 0 or idx >= len(image_samples):
        return JSONResponse({"error": "idx out of range"}, status_code=400)

    s = image_samples[idx]
    return {
        "image_name": s.get("image_name", ""),
        "image_url": s.get("image_url", ""),
        "text": s.get("text", "")
    }


@app.get("/image_qa")
def image_qa(idx: int = Query(...), q: str = Query("")):
    if idx < 0 or idx >= len(image_samples):
        return JSONResponse({"error": "idx out of range"}, status_code=400)

    if not q.strip():
        return {"error": "الرجاء إدخال السؤال."}

    sample = image_samples[idx]
    t0 = time.time()
    try:
        with infer_lock:
            answer = infer_with_prompt(build_image_prompt(sample, q.strip()))
        t1 = time.time()
        return {
            "answer": answer,
            "infer_sec": t1 - t0
        }
    except Exception as e:
        return {"error": repr(e)}

## 17. 启动服务
执行下面这个 cell 后，即可在板端启动网页应用。

In [30]:
import asyncio
import uvicorn

# 检查全局变量，防止重复启动
if "_server" not in globals(): _server = None

async def start_web_app(app, host="0.0.0.0", port=7860):
    global _server
    if _server:
        _server.should_exit = True
        await asyncio.sleep(1)
    
    config = uvicorn.Config(app=app, host=host, port=port, workers=1, log_level="info")
    _server = uvicorn.Server(config)
    asyncio.create_task(_server.serve())
    await asyncio.sleep(2)
    print(f"\n✅ 服务已启动: http://192.168.1.98:{port}")

await start_web_app(app, host="0.0.0.0", port=7860)

INFO:     Started server process [188000]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:7860 (Press CTRL+C to quit)



✅ 服务已启动: http://192.168.1.98:7860


## 19. 运行建议

推荐运行顺序：

1. 先执行环境初始化与模型加载  
2. 再执行 Audio / Image 单样本测试  
3. 最后执行启动服务 cell  
4. 浏览器访问：

```text
http://192.168.1.118:7861
```

## 提交建议

提交到仓库时建议目录如下：

```text
Online/arabic_multimodal_qa/
  arabic_multimodal_qa_all_in_one.ipynb
  README.md
```

## 29. 启动 Web 应用
如果前面的模型和 manifest 都已经准备好，可以直接启动网页。

In [31]:

print("在终端中运行：")
print("\n浏览器访问：")
print("http://192.168.1.98:7860")


在终端中运行：

浏览器访问：
http://192.168.1.98:7860


INFO:     192.168.1.75:52946 - "GET / HTTP/1.1" 200 OK
INFO:     192.168.1.75:52946 - "GET /audio HTTP/1.1" 200 OK
INFO:     192.168.1.75:52946 - "GET /audio_sample?idx=0 HTTP/1.1" 200 OK
INFO:     192.168.1.75:52946 - "GET /audio_qa?idx=0&q=%D9%85%D8%A7%20%D8%A7%D9%84%D9%85%D9%88%D8%B6%D9%88%D8%B9%20%D8%A7%D9%84%D8%B1%D8%A6%D9%8A%D8%B3%D9%8A%20%D9%84%D9%87%D8%B0%D8%A7%20%D8%A7%D9%84%D9%85%D9%82%D8%B7%D8%B9%20%D8%A7%D9%84%D8%B5%D9%88%D8%AA%D9%8A%D8%9F HTTP/1.1" 200 OK
